# exp12: Seleksi Gabungan dan Model Akhir

Mengevaluasi prosedur pemilihan pipeline dari bank kandidat terdaftar, memeriksa ketahanannya, lalu memilih konfigurasi dan melatih model akhir pada seluruh data. Notebook ini tidak menjalankan notebook lain.

Target memakai kesulitan tugas sebagai proxy cognitive load: easy=Low, medium=Medium, hard=High.

### Akses Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Environment

Menyiapkan library dari requirements-final.txt dan memakai fungsi bersama shared.experiments untuk menjaga konsistensi preprocessing dan evaluasi.

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import hashlib

ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE').resolve()
MODELS_DIR = ROOT / 'models'
manifest_path = MODELS_DIR / 'FINAL_MANIFEST.json'
if not manifest_path.is_file():
    raise RuntimeError('Upload FINAL_MANIFEST.json together with shared and requirements-final.txt')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
for relative, expected in manifest['files'].items():
    if relative.startswith('shared/') or relative == 'requirements-final.txt':
        target = MODELS_DIR / relative
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise RuntimeError(f'Incomplete or outdated upload: {relative}')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(MODELS_DIR / 'requirements-final.txt')])
import importlib.metadata
for module_name, package_name in [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('sklearn', 'scikit-learn'), ('xgboost', 'xgboost'), ('matplotlib', 'matplotlib'), ('joblib', 'joblib'), ('imblearn', 'imbalanced-learn'), ('catboost', 'catboost'), ('threadpoolctl', 'threadpoolctl')]:
    loaded = sys.modules.get(module_name)
    if loaded is not None and getattr(loaded, '__version__', None) != importlib.metadata.version(package_name):
        raise RuntimeError(f'Restart runtime before continuing: {module_name} was loaded before installation')


### Konfigurasi

Dataset utama adalah good_acceptable.csv. Mode full menjalankan evaluasi lengkap; seed utama 42. RESUME_DIR dapat diisi direktori run yang terputus. Dataset dan kode harus cocok dengan checkpoint.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

sys.path.insert(0, str(MODELS_DIR))
from shared.experiments.runner import Config, run
from shared.experiments.design import registry
from shared.data import load_dataset, describe_data, make_folds

EXPERIMENT = 'exp12'
MODE = 'full'
SEED = 42
RESUME_DIR = None
DATASET_PATH = ROOT / 'Dataset' / 'good_acceptable.csv'
OUTPUT_ROOT = MODELS_DIR / 'outputs'
config = Config(DATASET_PATH, OUTPUT_ROOT, EXPERIMENT, MODE, SEED)

### Dataset dan Kandidat

Sebelum training, periksa hasil full exp02 sampai exp11 yang memiliki dataset, manifest, mode, seed, dan runtime kompatibel. Bank mencakup kandidat terdaftar serta kombinasi terbatas clean/EAR/log dengan none/balanced/ROS 0,75/SMOTE 0,75 k=3 pada model referensi. Tabel menampilkan jumlah kandidat terdaftar sebelum deduplikasi; candidate_registry.json menyimpan bank akhir setelah pipeline identik dideduplicasi.

Validasi schema, distribusi kelas per peserta, dan manifest outer leave-one-participant-out dilakukan sebelum training. Identitas peserta/soal bukan input model.

In [ ]:
data = load_dataset(DATASET_PATH)
print(describe_data(data))
display(pd.crosstab(data.participant_id, data.difficulty_level))
folds = make_folds(data)
specs = registry(EXPERIMENT, MODE, SEED)
display(pd.DataFrame([{'model': s['id'], 'features': s['variant'], 'scaler': s['scaler'], 'registered_candidates': len(s['candidates']), 'inner_search': s['tune']} for s in specs]))
print('Outer folds:', len(folds))

### Preprocessing dan Training

Pada setiap outer fold, pilih pipeline menggunakan inner validation, bukan peringkat outer test. Gunakan kembali checkpoint identik; evaluasi inner yang belum tersedia dihitung di notebook ini. Uji estimator pipeline pilihan dengan seed 17 dan 2026, tanpa mengulang seluruh bank atau memilih seed test terbaik. Hitung bootstrap 1.000 kali pada tingkat peserta, sensitivitas subset test tidak overlap, hasil per kualitas, dan learning curve berdasarkan jumlah peserta training.

Checkpoint per unit menyimpan hasil lengkap dengan checksum. Progres mencakup outer fold, kandidat, inner fold, dan waktu proses.

In [ ]:
run_dir = run(config, resume_dir=RESUME_DIR)

### Evaluasi

Setelah evaluasi nested, pilih konfigurasi deployment melalui group validation seluruh dataset dan refit seluruh data. final_model.joblib menyimpan predictor beserta preprocessing; predict_frame menerima fitur tanpa target, dengan metadata/riwayat untuk kandidat temporal. Hasil model final pada training bukan skor generalisasi baru. RESULTS.md dan experiment_comparison.csv merangkum seluruh eksperimen, sementara prediksi/konfigurasi tersimpan untuk audit. Leaderboard tetap eksploratif dan bukan validasi eksternal.

Metrik utama adalah macro-F1 gabungan outer test. Periksa accuracy, balanced accuracy, precision/recall/F1 tiap kelas, dan hasil per peserta. Kelas tanpa support dilaporkan kosong. Gunakan full evaluation untuk laporan penelitian.

In [ ]:
summary = json.loads((run_dir / 'summary.json').read_text())
print(summary)
if summary['status'] != 'complete':
    raise RuntimeError('Incomplete evaluation')
if MODE == 'smoke':
    print('Smoke test only. Change MODE to full and run again for research evaluation.')
display(pd.read_csv(run_dir / 'metrics.csv'))
classes = pd.read_csv(run_dir / 'class_metrics.csv')
display(classes[classes.scope == 'pooled'])
display(pd.read_csv(run_dir / 'participant_metrics.csv'))
display(pd.read_csv(run_dir / 'search_results.csv'))
print('Saved results:', run_dir)

### Confusion Matrix

Baris menunjukkan kelas sebenarnya, kolom menunjukkan prediksi. Proporsi per baris pada diagonal menunjukkan recall kelas; periksa pertukaran Low, Medium, dan High.

In [ ]:
for name in pd.read_csv(run_dir / 'metrics.csv').model:
    display(Image(filename=str(run_dir / 'figures' / f'{name}_normalized.png')))

### Ringkasan dan Model Akhir

Ringkasan memisahkan perbandingan eksploratif, nested selection, sensitivitas seed, serta model akhir. Jangan memuat file joblib dari sumber yang tidak dipercaya.

In [ ]:
display(pd.read_csv(run_dir / 'experiment_comparison.csv'))
display(pd.read_csv(run_dir / 'seed_sensitivity.csv'))
from IPython.display import Markdown
display(Markdown((run_dir / 'RESULTS.md').read_text(encoding='utf-8')))
print('Final model:', run_dir / 'final_model.joblib')